# BaTiO₃ Phase 01 — Workbook and Computational-Data Audit (v3)

**Project:** Chemistry-aware scientific data curation and AI-ready workflow engineering  
**Research target:** Sr/La-substituted BaTiO₃, nominally `(Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃`  
**Method:** Read-only, reproducible, streaming audit of supplementary workbook sheets S1–S4.

**Purpose.** Determine whether published composition, descriptor and energy records can be mapped to auditable scientific identities **before** they are standardised, analysed or used to train a new ML model. Phase 01 does **not** rerun DFT calculations, evaluate capacitor performance or assert a defect mechanism.

**Research source:** [Published BaTiO₃ article](https://doi.org/10.1063/5.0314819) · [Official supplementary collection](https://doi.org/10.60893/figshare.jap.c.8234602)

**Eight workflow commitments:** (1) clarity in each cell, (2) minimal standard-library implementation, (3) deterministic outputs, (4) explicit setup and file paths, (5) safe input handling, (6) separation of evidence and assumptions, (7) built-in tests, and (8) findings tied to a follow-up decision.

**Only required source file:** `Supplementary Dataset.xlsx`, with numbered variants such as `Supplementary Dataset(2).xlsx` accepted. The journal PDF and supplementary DOCX provide scientific background but are not runtime dependencies. No data are downloaded automatically.

## 1 | Scientific scope: BaTiO₃ chemistry, DFT and ML evidence

BaTiO₃ is an `ABO₃` perovskite. Ba²⁺, Sr²⁺ and La³⁺ occupy the nominal **A-site composition space**; Ti is the **B-site element**. Substitution of Ba²⁺ by Sr²⁺ is **isovalent**; replacement by La³⁺ is **aliovalent** and raises questions about electronic and chemical charge compensation. A numerical Ti coefficient appearing in a source workbook is not an experimental determination of Ti vacancies.

**Evidence hierarchy:** a *candidate* composition (S1) is not a completed DFT calculation. The labelled S2/S4 rows are published computational results, not measurements. An S3 descriptor vector is an ML input representation, not a target or an experimentally validated material. This separation prevents confusing design proposals, calculations, predictions and physical measurements.

**Why audit first?** ML models can appear accurate while learning from incorrect joins, source inconsistencies or hidden target columns. Phase 01 checks the evidence connections before any modelling.

## 2 | Environment and source-file discovery

**Environment.** Python 3.11+ with JupyterLab and `ipykernel`. The audit computation uses **only Python's standard library** (no `pandas`, `openpyxl`, RDKit or web access). The existing Jupyter kernel `Python (BaTiO3 Phase00)` can be reused.

**Working-folder layouts accepted:**

```text
BaTiO3_Phase00/
├── BaTiO3_Phase01_FIXED_v3.ipynb
└── Supplementary Dataset.xlsx
```

or:

```text
BaTiO3_Phase00/
├── BaTiO3_Phase01_FIXED_v3.ipynb
└── data/
    └── raw/
        └── Supplementary Dataset(2).xlsx
```

To use another location, set `SOURCE_OVERRIDE` in the next code cell to an explicit path; keep it `None` for automatic discovery. The selected source remains read-only. Exact file bytes are identified using SHA-256; hashes verify identity, **not scientific validity**.

**Execution:** select the `Python (BaTiO3 Phase00)` kernel, then choose **Kernel → Restart Kernel and Run All Cells**. The same outputs overwrite previous versions with deterministic content on a successful run. A missing, ambiguous or invalid workbook is reported with an explicit **NOT COMPLETED** status; older outputs should not be treated as evidence of a new run.

**Important if you previously saw `NOT_COMPLETED_INPUT_OR_VALIDATION_ERROR`:** the notebook code can run but **the workbook audit cannot succeed without the Excel file**. In JupyterLab, keep your `.ipynb` and `Supplementary Dataset(2).xlsx` in the **same folder**, or set `SOURCE_OVERRIDE` to the correct absolute local path in the configuration cell. If you renamed or moved the notebook, check JupyterLab's file browser location. The notebook prints its current working directory and the locations searched. This repository does **not** distribute the source workbook.


In [4]:
"""Phase 01 | BaTiO3 published workbook source-data audit.

Standard-library-only, deterministic, streaming OOXML audit. Source material is
read-only and derived public outputs contain no substantial source rows.
"""
from __future__ import annotations

import csv
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation
from hashlib import sha256
from io import StringIO
from pathlib import Path
import os
import tempfile
import re
from xml.etree import ElementTree as ET
from zipfile import ZipFile, BadZipFile

NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
REL = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
PKG = "{http://schemas.openxmlformats.org/package/2006/relationships}"
SOURCE_NAME = "Supplementary Dataset.xlsx"
SHEETS = ("S1", "S2", "S3", "S4")
CHEM = ("Ba", "Sr", "La", "Ti")
E_ENERGY = ("Final energy", "formation energy")
TOL = Decimal("0.0000001")
KEY_STEP = Decimal("0.00000001")

# Optional local source override: None enables safe automatic discovery.
# Avoid committing private computer paths to public GitHub.
SOURCE_OVERRIDE = None

### Code stage 2A — File identification and cryptographic provenance

**What:** normalise download-counter suffixes, choose a workbook by a documented folder priority, and hash the source without modifying it.  
**Why:** a workbook is scientifically usable only if its specific version and provenance are known.  
**Outputs:** a selected source path held in memory; the SHA-256 appears in the written audit report. Multiple copies in the same folder are not silently resolved.

In [6]:
def file_key(name):
    p = Path(name)
    stem = re.sub(r"\s*\(\d+\)$", "", p.stem.strip())
    return re.sub(r"[^a-z0-9]", "", stem.casefold()) + p.suffix.casefold()

def find_source(base, explicit=None):
    """Select one XLSX deterministically, preferring the working folder.

    The most local source takes precedence over older copies in parent folders;
    multiple equally placed candidates remain an error (not a guessing game).
    An explicit path, if supplied, overrides discovery entirely.
    """
    if explicit is not None and str(explicit).strip():
        candidate = Path(explicit).expanduser().resolve()
        if not candidate.is_file():
            raise FileNotFoundError(
                f"PHASE 01 INPUT REQUIRED: configured workbook '{candidate.name}' is not a file. "
                "Check SOURCE_OVERRIDE or move the workbook to the working folder."
            )
        if candidate.suffix.casefold() != '.xlsx':
            raise ValueError("PHASE 01 INPUT FORMAT: the configured workbook must be an .xlsx file")
        return candidate
    base = Path(base).resolve()
    locations = [
        base, base / "data" / "raw", base.parent / "data" / "raw",
        base.parent, base / "Phase 00", base / "phase00_outputs",
    ]
    for folder in dict.fromkeys(locations):
        if not folder.is_dir():
            continue
        matches = sorted((p for p in folder.iterdir()
                          if p.is_file() and file_key(p.name) == file_key(SOURCE_NAME)),
                         key=lambda p:p.name.casefold())
        if len(matches) > 1:
            raise ValueError(
                "PHASE 01 AMBIGUOUS INPUT: multiple source workbooks in the same folder: "
                + ", ".join(p.name for p in matches)
                + ". Retain one copy here, or set SOURCE_OVERRIDE to the chosen file."
            )
        if matches:
            return matches[0]
    searched = "; ".join(str(folder) for folder in dict.fromkeys(locations))
    raise FileNotFoundError(
        "PHASE 01 INPUT REQUIRED: the publisher supplementary workbook was not found. "
        f"Jupyter working folder: {base}. Searched: {searched}. "
        "Put Supplementary Dataset.xlsx (or a numbered version such as "
        "Supplementary Dataset(2).xlsx) in the SAME folder as this notebook, "
        "or set SOURCE_OVERRIDE in the first configuration cell to its full local path. "
        "Do not treat previous phase01_outputs as new results."
    )

def digest_file(path):
    d = sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()

## 3 | Spreadsheet structure: source cells, formulas and cached values

Excel `.xlsx` files are ZIP packages containing XML. The parser extracts cell values, worksheet coordinates, original headings and any formulas. **S1 has more than 30,000 data rows**, so the audit streams one row at a time rather than loading a large workbook into a `pandas` DataFrame.

**Formula interpretation:** a formula cell can have a cached value, but the presence of a cache does **not** establish that the workbook recalculated it recently. The audit records formula counts and missing cached values; it never modifies the workbook or evaluates Excel formulas.

**Code output:** row-level information remains transient; only aggregate counts, header metadata and narrowly scoped labelled records are retained for checks.

In [8]:
def shared_strings(z):
    if "xl/sharedStrings.xml" not in z.namelist():
        return []
    strings = []
    with z.open("xl/sharedStrings.xml") as f:
        for _, elem in ET.iterparse(f, events=("end",)):
            if elem.tag == NS + "si":
                strings.append("".join(t.text or "" for t in elem.iter(NS + "t")))
                elem.clear()
    return strings

def workbook_paths(z):
    if "xl/workbook.xml" not in z.namelist():
        raise ValueError("INVALID XLSX: workbook structure is missing")
    book = ET.fromstring(z.read("xl/workbook.xml"))
    relroot = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
    relations = {x.attrib["Id"]: x.attrib["Target"] for x in relroot.findall(PKG + "Relationship")}
    result = {}
    for item in book.findall("./" + NS + "sheets/" + NS + "sheet"):
        name, rid = item.get("name"), item.get(REL + "id")
        target = relations[rid].replace("\\", "/")
        if target.startswith("/"):
            path = target.lstrip("/")
        elif target.startswith("xl/"):
            path = target
        else:
            from posixpath import normpath
            path = normpath("xl/" + target)
        if path not in z.namelist():
            raise ValueError(f"INVALID XLSX: missing worksheet payload for {name}")
        result[name] = path
    missing = set(SHEETS) - result.keys()
    if missing:
        raise ValueError("PHASE 01 MISSING WORKSHEETS: " + ", ".join(sorted(missing)))
    return result

def col_idx(ref):
    letters = re.match(r"[A-Za-z]+", ref or "")
    if not letters:
        raise ValueError(f"Unparseable cell reference: {ref!r}")
    n = 0
    for ch in letters.group().upper():
        n = n * 26 + ord(ch) - ord("A") + 1
    return n - 1

def excel_col(index):
    index += 1
    r = ""
    while index:
        index, rem = divmod(index - 1, 26)
        r = chr(65 + rem) + r
    return r

def parse_cell(el, strings):
    value_el, formula_el = el.find(NS + "v"), el.find(NS + "f")
    cell_type = el.get("t", "n")
    formula = formula_el is not None
    raw = value_el.text if value_el is not None else None
    if cell_type == "s" and raw is not None:
        try:
            raw = strings[int(raw)]
        except (ValueError, IndexError) as exc:
            raise ValueError("INVALID XLSX: shared-string reference failed") from exc
    elif cell_type == "inlineStr":
        raw = "".join(t.text or "" for t in el.iter(NS + "t"))
    if raw is None:
        return None, formula, cell_type
    return raw, formula, cell_type

def rows(z, sheet_path, strings):
    """Yield only one worksheet row at a time; memory independent of S1 row count."""
    with z.open(sheet_path) as fh:
        ctx = ET.iterparse(fh, events=("start", "end"))
        _, root = next(ctx)
        for event, item in ctx:
            if event != "end" or item.tag != NS + "row":
                continue
            row_id = int(item.get("r", "0"))
            values, formulas, types = {}, [], Counter()
            for cell in item.iter(NS + "c"):
                index = col_idx(cell.get("r"))
                value, formula, typ = parse_cell(cell, strings)
                values[index] = value
                types[typ] += 1
                if formula:
                    formulas.append((index, value is not None))
            yield row_id, values, formulas, types
            item.clear()
            root.clear()

## 4 | Composition identities and numerical comparison rules

**Identity construction:** the four tabulated coefficients `Ba`, `Sr`, `La` and `Ti` form a source-record join key, rounded to eight decimal places. This is a **practical matching convention**, not a claim that distinct thermodynamic phases or experimentally fabricated specimens are equivalent.

**Numeric comparison:** stored energies or descriptors are compared with `max(10⁻⁷ absolute, 10⁻⁹ relative)` tolerance. Missing, empty or non-finite values are *never* called successful matches. Later phases must verify the published formation-energy definition, reference states and units before making physical comparisons.

**Outputs:** deterministic matching keys and comparison functions—not standardised source data.

In [10]:
def number(value):
    if value is None or str(value).strip() == "":
        return None
    try:
        return Decimal(str(value).strip())
    except (ValueError, InvalidOperation):
        return None

def chemistry_key(row):
    vals = [number(row.get(k)) for k in CHEM]
    if any(x is None or not x.is_finite() for x in vals):
        return None
    return tuple(v.quantize(KEY_STEP) for v in vals)

def nearly(a, b):
    """Never count two missing / NaN scientific values as successful agreement."""
    if a is None or b is None or str(a).strip() == "" or str(b).strip() == "":
        return False
    x, y = number(a), number(b)
    if x is not None and y is not None:
        if not x.is_finite() or not y.is_finite():
            return False
        return abs(x - y) <= max(TOL, Decimal("1e-9") * max(abs(x), abs(y)))
    return str(a).strip() == str(b).strip()

def audit_status_for_issues(issues):
    if any(x['severity'] == 'ERROR' for x in issues):
        return 'BLOCKED_STRUCTURAL_ERRORS'
    if issues:
        return 'COMPLETED_WITH_REVIEW_ITEMS'
    return 'COMPLETED_NO_FLAGGED_ITEMS' 

def csv_text(headers, lines):
    out = StringIO(newline="")
    w = csv.writer(out, lineterminator="\n")
    w.writerow(headers)
    for line in lines:
        w.writerow(line)
    return out.getvalue()

## 5 | Phase 01 source audit — complete S1–S4 scan

**What the code verifies:** declared worksheet presence, header uniqueness, row and column counts, missing values, formula caches, source-cell types and composition-key uniqueness. It compares **S2 ↔ S3** descriptor values and **S2 ↔ S4** energy fields through matching chemical identities, not spreadsheet row positions. It also measures overlap with the S1 candidate population.

**Scientific rationale:** 231 DFT-labelled compositions are a limited training reference, and the larger S1 candidate space must not be mislabeled as 30,301 independently evaluated DFT cases. Joining records incorrectly would contaminate any ML evaluation.

**Critical error rules:** absent essential chemistry fields, duplicate identities, invalid DFT energy values and inconsistent cross-sheet comparisons generate **ERROR** findings. The overall status becomes `BLOCKED_STRUCTURAL_ERRORS`, so the audit cannot be mistaken for clean training data.

**Output:** an in-memory audit object containing source metadata, worksheet statistics, comparison summaries and structured review findings. Raw source rows are not exported.

In [12]:
def audit_workbook(path):
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Required workbook not found: {file_path.name}")
    result = {"source_filename": file_path.name, "source_sha256": digest_file(file_path),
              "source_bytes": file_path.stat().st_size, "sheets": {}, "issues": [],
              "cross_checks": [], "chemistry": {}, "column_records": [],
              "formulas": [], "source_data_status": "unmodified"}
    issues = result["issues"]
    def flag(section, severity, code, sheet, location, count, evidence, next_step):
        issues.append({"section":section, "severity":severity, "code":code,
                       "sheet":sheet,"location":location,"count":count,
                       "evidence":evidence,"action":next_step})
    try:
        with ZipFile(file_path) as z:
            if z.testzip() is not None:
                raise ValueError("INVALID XLSX: ZIP member integrity check failed")
            paths = workbook_paths(z)
            strings = shared_strings(z)
            kept = {}
            s1keys = set()
            for name in SHEETS:
                headers, rows_by_key = [], {}
                nulls = Counter(); formula_total = 0; no_cached = 0
                types = Counter(); observed = 0; n_rows = 0; duplicate_keys = 0
                missing_keys = 0; max_row = 0; incomplete_records = 0
                invalid_numbers = 0; unique_keys = set(); duplicate_headings = []
                for rownum, cells, formulas, cell_types in rows(z,paths[name],strings):
                    max_row = max(rownum, max_row)
                    if rownum == 1:
                        width = max(cells, default=-1) + 1
                        headers = [(str(cells.get(i)).strip() if cells.get(i) is not None else "") for i in range(width)]
                        duplicate_headings = [h for h,cnt in Counter(headers).items() if cnt > 1 or not h]
                        continue
                    if not any(v is not None and str(v) != "" for v in cells.values()):
                        continue
                    n_rows += 1
                    # Header coordinates are kept to trace anomalies; full S1 never retained.
                    row = {h: cells.get(j) for j,h in enumerate(headers)}
                    empty = sum(v is None or str(v).strip() == "" for v in row.values())
                    nulls.update(h for h,v in row.items() if v is None or str(v).strip()=="")
                    incomplete_records += (empty > 0)
                    types.update(cell_types)
                    observed += sum(v is not None and str(v) != "" for v in row.values())
                    formula_total += len(formulas)
                    no_cached += sum(not has_cache for _,has_cache in formulas)
                    key = chemistry_key(row)
                    if key is None:
                        missing_keys += 1
                    elif key in unique_keys:
                        duplicate_keys += 1
                        if duplicate_keys <= 3:
                            flag("identity", "ERROR", "DUPLICATE_COMPOSITION", name, f"row {rownum}", 1,
                                 "More than one row maps to the same Ba/Sr/La/Ti key",
                                 "Inspect source coefficients and identity rules")
                    else:
                        unique_keys.add(key)
                    if name == "S1":
                        if key is not None: s1keys.add(key)
                    elif key is not None:
                        # Retain 231 labelled rows, not S1's 30,301 rows.
                        rows_by_key.setdefault(key, (rownum, row))
                if not headers or n_rows == 0:
                    flag("structure", "ERROR", "EMPTY_WORKSHEET", name,"worksheet",1,
                         "No header/data rows were discovered","Verify workbook sheet and source version")
                if duplicate_headings:
                    flag("structure", "ERROR", "DUPLICATE_OR_BLANK_HEADERS", name,"row 1",len(duplicate_headings),
                         "A source header is blank or non-unique","Review the source schema")
                required_headings = set(CHEM) | (set(E_ENERGY) if name in ('S2', 'S4') else set())
                missing_headers = sorted(required_headings - set(headers))
                if missing_headers:
                    flag("structure", "ERROR", "REQUIRED_FIELDS_ABSENT", name, "row 1", len(missing_headers),
                         "Missing declared scientific fields: " + ", ".join(missing_headers),
                         "Reconcile source version and field labels before further comparison")
                if name in ('S2', 'S4'):
                    invalid_energy_count = 0
                    for _, scientific_row in rows_by_key.values():
                        for energy_name in E_ENERGY:
                            energy_value = number(scientific_row.get(energy_name))
                            if energy_value is None or not energy_value.is_finite():
                                invalid_energy_count += 1
                    if invalid_energy_count:
                        flag("target", "ERROR", "INVALID_ENERGY_VALUES", name, "energy fields", invalid_energy_count,
                             "One or more DFT energy values are missing, nonnumeric or nonfinite",
                             "Inspect numerical cell values; do not treat missing targets as model labels")
                if missing_keys:
                    flag("identity", "ERROR", "UNREADABLE_COMPOSITION", name,"composition columns",missing_keys,
                         "At least one key coefficient is missing or not numeric", "Inspect affected original rows")
                if no_cached:
                    flag("formula", "WARNING", "UNCACHED_FORMULA", name,"formula cells",no_cached,
                         "Formula cells have no stored cached result", "Recalculate in Excel for a separate source copy; preserve original")
                if incomplete_records:
                    flag("missingness", "WARNING", "INCOMPLETE_ROWS",name,"data area",incomplete_records,
                         "At least one data row contains an empty declared field", "Review blank-cell semantics in source")
                expected = {"S1":100,"S2":102,"S3":82,"S4":10}[name]
                if len(headers)!=expected:
                    flag("structure","WARNING","UNEXPECTED_WIDTH",name,"row 1",len(headers),
                         f"Observed width {len(headers)}; README preliminary width {expected}","Check supplementary data version")
                result["sheets"][name] = {"data_rows":n_rows,"columns":len(headers),"headings":headers,
                    "empty_cells":sum(nulls.values()), "rows_with_empty_cells":incomplete_records,
                    "empty_by_column":dict(nulls),"formula_cells":formula_total,
                    "formula_without_cached":no_cached,"types":dict(types),
                    "unique_composition_keys":len(unique_keys),"duplicate_composition_rows":duplicate_keys,
                    "missing_composition_keys":missing_keys,"observed_nonblank_cells":observed,
                    "max_excel_row":max_row}
                kept[name] = rows_by_key
                for i,h in enumerate(headers):
                    n_null = nulls.get(h,0)
                    role = ("nominal composition coefficient" if h in CHEM else
                            "oxygen coefficient" if h=="O" else
                            "reported DFT energy / reference target; unit requires source methods" if h in E_ENERGY else
                            "calculation method" if h=="Method" else
                            "crystal-structure label" if h=="Space Group" else
                            "source row identifier (not join key)" if h=="NO." else
                            "published engineered descriptor; definition requires supplement")
                    result["column_records"].append((name,excel_col(i),h,role,n_rows-n_null,n_null,
                                                       "phase01_source_audit"))
                result["formulas"].append((name,formula_total,no_cached,
                    "Cached results inspected, formulas not recalculated" if formula_total else "No formula cells observed"))
            # Source-to-source comparisons are keyed on composition, not row alignment.
            keys = {name:set(kept[name]) for name in ("S2","S3","S4")}
            for a,b in (("S2","S3"),("S2","S4"),("S3","S4")):
                intersection = keys[a] & keys[b]
                result["cross_checks"].append((f"{a} to {b}","composition key coverage",len(intersection),
                     len(keys[a]-keys[b]),len(keys[b]-keys[a]),0,"Ba/Sr/La/Ti composite key"))
                if keys[a] != keys[b]:
                    flag("linkage","ERROR","KEY_MISMATCH",f"{a}/{b}","composition keys",
                         len(keys[a]^keys[b]),"Composition sets differ between worksheets",
                         "Check missing or differently encoded compositions")
            for a,b,columns in (("S2","S3",[x for x in result["sheets"]["S3"]["headings"] if x not in CHEM]),
                               ("S2","S4",list(E_ENERGY))):
                both = sorted(keys[a]&keys[b]); compared=0; mismatches=0
                examples=[]
                for key in both:
                    ra, rb = kept[a][key], kept[b][key]
                    for field in columns:
                        if field not in ra[1] or field not in rb[1]:
                            flag("linkage", "ERROR","MISSING_COMPARE_FIELD", f"{a}/{b}",field,1,
                                 "Comparison field missing from workbook", "Review header mapping")
                            continue
                        compared += 1
                        if not nearly(ra[1][field],rb[1][field]):
                            mismatches += 1
                            if len(examples)<4: examples.append(f"{a}!{field} row {ra[0]} vs {b} row {rb[0]}")
                result["cross_checks"].append((f"{a} to {b}","matched values",compared,0,0,mismatches,
                     f"{len(columns)} declared comparable fields; tolerance max(1e-7, relative 1e-9)"))
                if mismatches:
                    flag("linkage","ERROR","VALUE_MISMATCH",f"{a}/{b}","; ".join(examples),mismatches,
                         "One or more shared descriptor/energy values disagree", "Investigate exact source cells")
            s2keys = keys["S2"]
            result["cross_checks"].append(("S2 to S1","labelled key present among S1 candidates",
                len(s2keys & s1keys),len(s2keys - s1keys),0,0,"Presence is not evidence of DFT labels for all S1 rows"))
            if s2keys - s1keys:
                flag("linkage","WARNING","S2_OUTSIDE_S1","S2/S1","compositions",len(s2keys-s1keys),
                     "Labelled compositions absent from candidate sheet", "Review published candidate domain")
            s2 = kept["S2"]
            comp = {"n":len(s2),"a_site_tested":0,"a_site_violations":0,"ti_law_tested":0,
                    "ti_law_matches":0,"ti_below_one":0,"ti_deviation_from_one":0,"la_positive":0,
                    "la_positive_ti_below_one":0}
            for key,(rn,row) in s2.items():
                ba,sr,la,ti=[number(row.get(h)) for h in CHEM]
                if all(v is not None and v.is_finite() for v in (ba,sr,la,ti)):
                    comp["a_site_tested"]+=1
                    if abs(ba+sr+la-1)>TOL or min(ba,sr,la)<-TOL:
                        comp["a_site_violations"]+=1
                    comp["ti_law_tested"]+=1
                    comp["ti_law_matches"]+=(abs(ti-(1-la/2))<=TOL)
                    comp["ti_below_one"]+=(ti<1-TOL)
                    comp["ti_deviation_from_one"]+=(abs(ti-1)>TOL)
                    comp["la_positive"]+=(la>TOL)
                    comp["la_positive_ti_below_one"]+=(la>TOL and ti<1-TOL)
            result["chemistry"] = comp
            if comp["a_site_violations"]:
                flag("chemistry","WARNING","A_SITE_FRACTION", "S2", "Ba/Sr/La", comp["a_site_violations"],
                     "Reported A-site sum or sign differs from nominal constraints", "Review source encoding")
            if comp["ti_below_one"]:
                flag("chemistry","REVIEW","TI_LA_REPRESENTATION","S2","Ti/La coefficients",comp["ti_below_one"],
                     f"Ti < 1 for {comp['ti_below_one']} records; Ti=1−La/2 matches {comp['ti_law_matches']} of {comp['ti_law_tested']}",
                     "Reconcile spreadsheet coefficient convention with VCA and source methods; do not infer vacancies")
            # Expected counts are a check of source-version consistency, not hard requirements.
            for name, count in (("S1",30301),("S2",231),("S3",231),("S4",231)):
                actual = result["sheets"][name]["data_rows"]
                if actual != count:
                    flag("structure","WARNING","COUNT_DIFFERS_FROM_README",name,"data area",abs(actual-count),
                         f"Observed {actual}; preliminary README count {count}","Check source version and records")
            if "formation energy" in result["sheets"]["S3"]["headings"]:
                flag("ML","ERROR","TARGET_LEAKAGE_FIELD","S3","header",1,
                     "Formation-energy target appears among the S3 input columns", "Exclude target before any modelling")
            result["status"] = audit_status_for_issues(issues)
            return result
    except BadZipFile as exc:
        raise ValueError("PHASE 01 INVALID INPUT: not a valid XLSX ZIP container") from exc
    except ET.ParseError as exc:
        raise ValueError("PHASE 01 INVALID INPUT: workbook XML is malformed") from exc
    except KeyError as exc:
        raise ValueError("PHASE 01 INVALID INPUT: a required OOXML worksheet component is missing") from exc

## 6 | Chemical-representation diagnostic — the Ti–La question

The audit inspects two relationships in the **published tabulated coefficients**. These are *source-data diagnostic checks*, not established chemical defect equations.

**A-site fraction sum:**

> **Ba + Sr + La ≈ 1**

**Observed Ti–La coefficient pattern:**

> **Ti ≈ 1 − (La / 2)**

Both equations use ordinary Unicode symbols rather than rendering-dependent LaTeX so they display reliably in **JupyterLab, exported HTML and GitHub Markdown**, including when MathJax is unavailable.

**Chemistry explanation.** In the nominal perovskite `(Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃`, the Ba/Sr/La fractions sum to one A-site. Replacing Ba²⁺ with Sr²⁺ is isovalent; replacing Ba²⁺ with La³⁺ is aliovalent and raises charge-accounting questions. The second relationship is observed in the *reported workbook coefficients* and must **not** be interpreted on its own as experimental proof of titanium vacancies. The source paper describes computational treatment including the virtual crystal approximation (VCA); method context must be reviewed before interpreting these values. This audit preserves the original source coefficients and never silently substitutes a different composition formula.

**Decision and next research action:** retain the Ti–La pattern as a `REVIEW` finding, compare it with the article's VCA and charge-compensation descriptions, and document the chosen representation before defining the Phase 02 schema. Phase 01 does not independently establish DFT convergence, physical defect populations, dielectric performance or ML generalisation.


## 7 | Transparent research reports and deterministic exports

**Outputs written only after a successful source parse:**

| File (in `phase01_outputs/`) | Interpretation and next use |
|---|---|
| `workbook_inventory.csv` | S1–S4 size, missingness, formula counts and composition uniqueness |
| `column_inventory.csv` | Original headings, column locations and *preliminary* scientific roles |
| `formula_inventory.csv` | Formula and cached-value status; no recalculation |
| `cross_sheet_checks.csv` | Keyed S2/S3/S4 matches, numerical comparisons and S1 overlap |
| `qc_findings.csv` | Severity, evidence and **action required** for each issue |
| `source_audit_report.md` | Scientific interpretation, limitations and review priorities |

**Privacy/rights boundary:** outputs contain aggregation and source-provenance metadata, not a redistributed copy of the publisher's full labelled dataset. Original journal files remain local unless licence terms explicitly permit redistribution. A successful structural audit is not experimental or independent DFT validation.

**Data integrity:** outputs are first staged as complete files, then replaced. Re-running with identical source bytes yields identical output bytes and SHA-256 digests. Unexpected failures are reported without exporting misleading new results.

In [15]:
def report_markdown(r):
    s=r["sheets"]; c=r["chemistry"]
    lines=["# Phase 01 — Computational workbook source audit", "",
      "## Evidence boundary and audit status", "",
      f"**Audit status:** `{r['status']}`. A blocked status indicates structural or scientific data errors that require correction before downstream use.", "",
      "This document records a read-only, structural audit of the published supplementary workbook. "
      "Matching source values does not establish DFT convergence, experimental ceramic synthesis, "
      "ferroelectric response or capacitor reliability.", "",
      "## Source identity", "",f"- Workbook: `{r['source_filename']}`",f"- SHA-256: `{r['source_sha256']}`",
      f"- Byte size: {r['source_bytes']:,}","- Source modifications: none", "",
      "## Worksheet inventory", "", "| Sheet | Data rows | Columns | Unique composition keys | Blank cells | Formula cells |",
      "|---|---:|---:|---:|---:|---:|"]
    for n in SHEETS:
        x=s[n];lines.append(f"| {n} | {x['data_rows']:,} | {x['columns']} | {x['unique_composition_keys']:,} | {x['empty_cells']:,} | {x['formula_cells']:,} |")
    lines += ["", "**Interpretation.** S1 is a candidate/descriptors space, not a set of 30,301 independently verified DFT labels. "
             "S2 and S4 store reference-energy fields; S3 is the 82-column selected feature representation.","",
             "## Keyed linkage comparisons", "",
             "| Comparison | Check | Matches/comparisons | Missing in right | Missing in left | Value mismatches |",
             "|---|---|---:|---:|---:|---:|"]
    for a,b,comp,missr,missl,mismatch,detail in r['cross_checks']:
        lines.append(f"| {a} | {b} | {comp:,} | {missr:,} | {missl:,} | {mismatch:,} |")
    lines += ["", "Comparisons use Ba/Sr/La/Ti identity keys rather than worksheet row numbers. "
              "Numeric equality allows absolute tolerance 1×10⁻⁷ or relative tolerance 1×10⁻⁹, whichever is greater.","",
              "## Crystal-chemistry diagnostics", "",
              f"- S2 compositions assessed: {c['n']}.",
              f"- A-site sum/sign tests: {c['a_site_tested']}; violations: {c['a_site_violations']}.",
              f"- Ti coefficient below 1: {c['ti_below_one']}.",
              f"- Ti = 1 − La/2 within tolerance: {c['ti_law_matches']} of {c['ti_law_tested']}.",
              f"- Positive La coefficients: {c['la_positive']}.","",
              "The nominal A-site formula (Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃ is not a warrant to overwrite the workbook's "
              "reported Ti coefficients. Sr²⁺ substitution is isovalent; La³⁺ substitution is aliovalent. "
              "The pattern in tabulated Ti values raises a **representation and charge-accounting question**. "
              "It cannot, on its own, establish real Ti vacancies; the computational work uses a documented VCA treatment.","",
              "## Issues and next actions", "",
              "Severity meanings: **ERROR** blocks downstream readiness; **WARNING** requires inspection; **REVIEW** records an unresolved scientific interpretation.", ""]
    if r['issues']:
        for x in r['issues']:
            lines.append(f"- **{x['severity']} — {x['code']}** ({x['sheet']}, {x['location']}): "
                         f"{x['evidence']}. **Follow-up:** {x['action']}.")
    else:
        lines.append("- No automated structural issues flagged; scientific validation remains incomplete.")
    lines += ["", "## Phase boundaries and open decisions", "",
       "- Phase 01 validates spreadsheet structure, composition identity and cross-sheet agreement only.",
       "- Formation-energy units and reference states require separate source-method reconciliation.",
       "- DFT convergence, VCA adequacy, defect chemistry, device manufacturing and dielectric performance have not been independently verified.",
       "- Phase 02 will define scientific data semantics; Phase 03 will expand chemical QC and record-level exception review.",
       "- Public GitHub release should omit the original publisher workbook and review any row-level extracts for licensing/redistribution compliance.", ""]
    return "\n".join(lines)

def prepare_exports(r):
    inv=[]
    for n in SHEETS:
        x=r['sheets'][n]
        inv.append((n,x['data_rows'],x['columns'],x['unique_composition_keys'],x['duplicate_composition_rows'],
                    x['missing_composition_keys'],x['empty_cells'],x['rows_with_empty_cells'],
                    x['formula_cells'],x['formula_without_cached'],x['observed_nonblank_cells'],
                    "candidate descriptors" if n=='S1' else "DFT-labelled descriptors and energies" if n=='S2' else
                    "selected ML input descriptors" if n=='S3' else "reported method/structure and energies"))
    outputs={
       "workbook_inventory.csv": csv_text(("sheet","data_rows","columns","unique_composition_keys","duplicate_composition_rows","unreadable_composition_rows","empty_cells","rows_with_empty_cells","formula_cells","uncached_formula_cells","nonblank_cells","scientific_role"),inv),
       "column_inventory.csv": csv_text(("sheet","excel_column","source_heading","preliminary_scientific_role","nonblank_rows","empty_rows","audit_status"),r['column_records']),
       "formula_inventory.csv": csv_text(("sheet","formula_cells","missing_cached_results","interpretation"),r['formulas']),
       "cross_sheet_checks.csv": csv_text(("comparison","check","matches_or_comparisons","missing_right","missing_left","value_mismatches","method_or_note"),r['cross_checks']),
       "qc_findings.csv":csv_text(("section","severity","code","sheet","location","count","evidence","action"),
                      ((x[k] for k in ("section","severity","code","sheet","location","count","evidence","action")) for x in r['issues'])),
       "source_audit_report.md":report_markdown(r),
    }
    return outputs

def write_exports(r, outdir):
    """Stage outputs first, then replace named files; same inputs -> same bytes."""
    outdir = Path(outdir)
    payloads = prepare_exports(r)
    # Avoid writing outputs if export generation itself failed.
    outdir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".phase01_staging_", dir=outdir) as scratch:
        staging = Path(scratch)
        for name, content in sorted(payloads.items()):
            (staging / name).write_bytes(content.encode("utf-8"))
        for name in sorted(payloads):
            os.replace(staging / name, outdir / name)
    return {name: sha256(content.encode("utf-8")).hexdigest()
            for name, content in sorted(payloads.items())}

## 8 | Built-in validation and regression tests

**What:** run offline checks for filename handling, worksheet coordinates, composition precision, missing and invalid numerics, formula caches, CSV escaping, ambiguous sources and audit-blocking status.  
**Why:** predictable behavior is more reliable than merely assuming the code runs.  
**Pass criterion:** every functional test passes; any failed test makes the audit **NOT COMPLETED**.  
**Output:** a named list of successful checks, separate from any scientific finding.

In [17]:
def self_tests():
    """Quick offline engineering tests; no external source files or network needed."""
    tests = []
    def check(name, condition):
        if not condition:
            raise AssertionError(f"ENGINEERING TEST FAILED: {name}")
        tests.append(name)
    check("Numbered download suffix accepted",
          file_key('Supplementary Dataset(2).xlsx') == file_key(SOURCE_NAME))
    check("Different workbook names remain distinct", file_key('Other Dataset.xlsx') != file_key(SOURCE_NAME))
    check("Excel A maps to zero", col_idx('A71') == 0)
    check("Excel CX maps to 101", col_idx('CX1') == 101)
    check("Excel column conversion round-trip", excel_col(101) == 'CX')
    check("Composition key normalises numeric strings",
          chemistry_key({'Ba':'0.95','Sr':'0','La':'0.05','Ti':'0.975'}) ==
          chemistry_key({'Ba':'0.950000000','Sr':'0','La':'0.050000000','Ti':'0.97500'}))
    check("Missing coefficient blocks linkage",
          chemistry_key({'Ba':'1','Sr':'0','La':None,'Ti':'1'}) is None)
    check("Numerical roundoff tolerated", nearly('1.00000000001','1.0'))
    check("Material numerical disagreement flagged", not nearly('1.01','1.0'))
    check("Missing values not a successful scientific match", not nearly(None,None))
    check("Missing-versus-number not a match", not nearly(None,'0.0'))
    check("Infinity never matches infinity", not nearly('Infinity','Infinity'))
    check("Nonnumeric NaN values not successful scientific matches", not nearly('NaN','NaN'))
    check("CSV safely quotes comma fields", '"a,b"' in csv_text(['x'],[['a,b']]))
    check("Empty QC export has header", csv_text(['code'],[]).strip() == 'code')
    check("Formula cache absence is explicitly distinguishable",
          parse_cell(ET.fromstring('<c xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" r="A1"><f>1+1</f></c>'),[])[1:] == (True,'n'))
    check("Structural errors block downstream use",
          audit_status_for_issues([{'severity':'ERROR'}]) == 'BLOCKED_STRUCTURAL_ERRORS')
    check("Scientific reviews stay distinct from failure",
          audit_status_for_issues([{'severity':'REVIEW'}]) == 'COMPLETED_WITH_REVIEW_ITEMS')
    with tempfile.TemporaryDirectory() as folder:
        temp = Path(folder)
        missing_raised = False
        try:
            find_source(temp)
        except FileNotFoundError as exc:
            missing_raised = 'PHASE 01 INPUT REQUIRED' in str(exc)
        check("Missing publisher workbook has actionable error", missing_raised)
        (temp/'Supplementary Dataset.xlsx').write_bytes(b'example')
        (temp/'Supplementary Dataset(2).xlsx').write_bytes(b'example')
        duplicate_raised = False
        try:
            find_source(temp)
        except ValueError as exc:
            duplicate_raised = 'AMBIGUOUS INPUT' in str(exc)
        check("Ambiguous workbooks require explicit selection", duplicate_raised)
        (temp/'Supplementary Dataset(2).xlsx').unlink()
        check("Single workbook discovered predictably",
              find_source(temp).name == 'Supplementary Dataset.xlsx')
        (temp/'data'/'raw').mkdir(parents=True)
        (temp/'data'/'raw'/'Supplementary Dataset(4).xlsx').write_bytes(b'example')
        check("Current-folder source takes precedence over nested backup",
              find_source(temp).name == 'Supplementary Dataset.xlsx')
        invalid_raised = False
        try:
            audit_workbook(temp/'Supplementary Dataset.xlsx')
        except ValueError as exc:
            invalid_raised = 'INVALID INPUT' in str(exc)
        check("Non-XLSX data rejected with helpful message", invalid_raised)
    return tests

## 9 | Restart-safe execution and run status

Run every cell in order using **Restart Kernel and Run All Cells**. No imports from a separate `.py` file are required.

This execution cell explicitly initialises its state. Anticipated problems (missing input, duplicate input files, invalid workbook, permission errors, validation-test failures) generate a clear **NOT COMPLETED** message instead of causing a second cascade of undefined-variable errors. **No code can guarantee error-free execution in every environment**; unexpected failures remain visible and should be investigated. Incomplete attempts do not represent new scientific results.

In [19]:
# Initialise state so every downstream cell can report a meaningful result.
RUN_STATUS = "NOT_STARTED"
RUN_ERROR = None
TEST_NAMES = []
AUDIT = None
SOURCE = None
OUTPUT_HASHES = {}
OUTPUT_DIR = Path.cwd() / "phase01_outputs"

try:
    print(f"JUPYTER WORKING FOLDER: {Path.cwd()}")
    print("SOURCE CONFIGURATION: " + (str(SOURCE_OVERRIDE) if SOURCE_OVERRIDE else "automatic discovery"))
    TEST_NAMES = self_tests()
    print(f"ENGINEERING TESTS: PASS ({len(TEST_NAMES)} checks)")

    SOURCE = find_source(Path.cwd(), explicit=SOURCE_OVERRIDE)
    print(f"SOURCE SELECTED: {SOURCE.name} | size={SOURCE.stat().st_size:,} bytes")

    AUDIT = audit_workbook(SOURCE)
    # Check that input bytes have not changed between first hash and export.
    if digest_file(SOURCE) != AUDIT["source_sha256"]:
        raise ValueError("Source changed during audit; rerun after keeping the workbook unchanged")

    OUTPUT_HASHES = write_exports(AUDIT, OUTPUT_DIR)
    RUN_STATUS = AUDIT["status"]
    print(f"AUDIT STATUS: {RUN_STATUS}")
    print(f"OUTPUTS: {len(OUTPUT_HASHES)} files generated in phase01_outputs/")
    for filename in sorted(OUTPUT_HASHES):
        print("  +", filename)
    if RUN_STATUS == "BLOCKED_STRUCTURAL_ERRORS":
        print("ACTION REQUIRED: structural ERROR findings block downstream use; inspect qc_findings.csv")
except (FileNotFoundError, ValueError, AssertionError, PermissionError, OSError, ET.ParseError, KeyError) as exc:
    RUN_STATUS = "NOT_COMPLETED_INPUT_OR_VALIDATION_ERROR"
    RUN_ERROR = f"{type(exc).__name__}: {exc}"
    print("PHASE 01 STATUS:", RUN_STATUS)
    print("REASON:", RUN_ERROR)
    print("ACTION: verify the source workbook, folder, read permissions and notebook test messages.")
    print("IMPORTANT: any pre-existing phase01_outputs/ files are from an earlier run.")
except Exception as exc:
    # Explicitly distinguish an unexpected programming/system defect from a valid result.
    RUN_STATUS = "NOT_COMPLETED_UNEXPECTED_ERROR"
    RUN_ERROR = f"{type(exc).__name__}: {exc}"
    print("PHASE 01 STATUS:", RUN_STATUS)
    print("UNEXPECTED ERROR:", RUN_ERROR)
    print("ACTION: inspect notebook diagnostics and correct the failure before citing outputs.")
    print("IMPORTANT: any pre-existing phase01_outputs/ files are from an earlier run.")

JUPYTER WORKING FOLDER: [local Phase 01 Jupyter workspace]
SOURCE CONFIGURATION: automatic discovery
ENGINEERING TESTS: PASS (23 checks)
SOURCE SELECTED: Supplementary Dataset.xlsx | size=27,198,367 bytes
AUDIT STATUS: COMPLETED_WITH_REVIEW_ITEMS
OUTPUTS: 6 files generated in phase01_outputs/
  + column_inventory.csv
  + cross_sheet_checks.csv
  + formula_inventory.csv
  + qc_findings.csv
  + source_audit_report.md
  + workbook_inventory.csv


## 10 | Interpreting results and deciding what happens next

The status below is part of the scientific record:

- `COMPLETED_NO_FLAGGED_ITEMS`: declared automated structural checks finished without flagged issues; this **does not** prove DFT correctness.
- `COMPLETED_WITH_REVIEW_ITEMS`: checks completed, but reviewable chemical/data questions remain.
- `BLOCKED_STRUCTURAL_ERRORS`: the audit completed but identified structural errors; **do not** treat its records as validated ML inputs.
- `NOT_COMPLETED_...`: notebook execution did not establish a valid new audit. Correct the reported problem; do not publish old outputs as current results.

**Next research actions:** investigate the Ti–La representation against computational methods, verify formation-energy definitions and units, explain S1/S2 candidate coverage, then construct a source-aware scientific schema and record-level QC (Phases 02–03).

In [21]:
print("\n" + "=" * 66)
print("PHASE 01 — FINAL STATUS:", RUN_STATUS)
print("=" * 66)
if AUDIT is not None:
    for name in SHEETS:
        item = AUDIT['sheets'][name]
        print(f"{name}: {item['data_rows']:,} rows | {item['columns']} columns | "
              f"{item['unique_composition_keys']:,} unique keys | "
              f"{item['formula_cells']:,} formula cells")
    for comparison, label, count, missing_right, missing_left, mismatches, _ in AUDIT['cross_checks']:
        print(f"{comparison} / {label}: compared={count:,}, "
              f"missing_right={missing_right}, missing_left={missing_left}, mismatches={mismatches}")
    c = AUDIT['chemistry']
    print(f"A-site constraint violations: {c['a_site_violations']}/{c['a_site_tested']}")
    print(f"Ti < 1: {c['ti_below_one']}/{c['ti_law_tested']}")
    print(f"Ti = 1 - La/2 agreement: {c['ti_law_matches']}/{c['ti_law_tested']}")
    for item in AUDIT['issues']:
        print(f"{item['severity']} | {item['code']} | count={item['count']} | NEXT: {item['action']}")
    print("EVIDENCE LIMIT: checks of the published workbook, not independent DFT or experiment.")
else:
    print("No valid new scientific audit was produced.")
    print("REQUIRED ACTION:", RUN_ERROR or "inspect the preceding cell")


PHASE 01 — FINAL STATUS: COMPLETED_WITH_REVIEW_ITEMS
S1: 30,301 rows | 100 columns | 30,301 unique keys | 30,301 formula cells
S2: 231 rows | 102 columns | 231 unique keys | 0 formula cells
S3: 231 rows | 82 columns | 231 unique keys | 0 formula cells
S4: 231 rows | 10 columns | 231 unique keys | 0 formula cells
S2 to S3 / composition key coverage: compared=231, missing_right=0, missing_left=0, mismatches=0
S2 to S4 / composition key coverage: compared=231, missing_right=0, missing_left=0, mismatches=0
S3 to S4 / composition key coverage: compared=231, missing_right=0, missing_left=0, mismatches=0
S2 to S3 / matched values: compared=18,018, missing_right=0, missing_left=0, mismatches=0
S2 to S4 / matched values: compared=462, missing_right=0, missing_left=0, mismatches=0
S2 to S1 / labelled key present among S1 candidates: compared=21, missing_right=210, missing_left=0, mismatches=0
A-site constraint violations: 0/231
Ti < 1: 210/231
Ti = 1 - La/2 agreement: 231/231
WARNING | S2_OUTSI

## 11 | Scientific conclusion and release boundary

**Verified by this workflow when successfully executed:** exact workbook identity, S1–S4 structure, source-column inventory, missingness, formula-cache statistics, Ba/Sr/La/Ti composition keys, key-based S2/S3/S4 comparisons and declared chemistry diagnostics.

**Requires review:** why the source workbook uses its Ti–La coefficient pattern, how the published VCA handles charge, differences between candidate and labelled composition spaces, and whether any outlying source values have an alternative interpretation.

**Not verified:** independently converged DFT results, numerical energy reference consistency beyond stored-value matching, experimental BaTiO₃ processing, vacancy concentrations, dielectric performance, ML out-of-domain predictive accuracy or capacitor reliability.

**Release discipline:** keep publisher source workbooks out of public GitHub unless their licence allows redistribution. Publish only inspected notebook code, HTML summary, aggregate CSVs and the scientific report, with numerical units and evidence classes explicit. The source notebook and generated reports contain no additional personal attribution.